# Data Analysis - Capa GOLD

En este notebook se realiza la transformación de los datos de la capa SILVER para la creación de información analítica.

El objetivo es generar información agregada para el análisis de clientes y sus gastos.

In [1]:
import os

# Configurar variables de entorno para Hadoop en Windows
HADOOP_HOME = os.environ.get("HADOOP_HOME", "C:\\hadoop")

os.environ["HADOOP_HOME"] = HADOOP_HOME
os.environ["hadoop.home.dir"] = HADOOP_HOME
os.environ["PATH"] = f"{os.environ.get('PATH', '')};{HADOOP_HOME}\\bin"

In [2]:
# Importar librerías

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    count,
    sum,
    avg,
    min,
    max,
    round,
    countDistinct
)

## Crear la sesión de Apache Spark

Se crea una sesión de Spark para el procesamiento distribuido de datos.

In [3]:
# Crear sesión de Spark

spark = (
    SparkSession.builder
    .appName("FinancialDigitalTwin_Gold")
    .getOrCreate()
)

print("Spark inicializado")

Spark inicializado


## Cargar los conjuntos de datos

In [4]:
# Cargar datasets

INPUT_PATH = "../data/processed/silver"
OUTPUT_PATH = "../data/processed/gold"

import os

os.makedirs(OUTPUT_PATH, exist_ok=True)

users_spark = (
    spark.read
    .parquet(f"{INPUT_PATH}/users.parquet")
)

cards_spark = (
    spark.read
    .parquet(f"{INPUT_PATH}/cards.parquet")
)

transactions_spark = (
    spark.read
    .parquet(f"{INPUT_PATH}/transactions.parquet")
)

print("Datos cargados")

Datos cargados


## Explorar la información

In [5]:
# Mostrar primeras filas

users_spark.show(5)
cards_spark.show(5)
transactions_spark.show(5)

+---+-----------+--------------+----------+-----------+------+--------------------+--------+---------+-----------------+-------------+----------+------------+----------------+
| id|current_age|retirement_age|birth_year|birth_month|gender|             address|latitude|longitude|per_capita_income|yearly_income|total_debt|credit_score|num_credit_cards|
+---+-----------+--------------+----------+-----------+------+--------------------+--------+---------+-----------------+-------------+----------+------------+----------------+
|  0|         33|            69|      1986|          3|  male|     858 plum avenue|   43.59|   -70.33|          29237.0|      59613.0|   36199.0|         763|               4|
|  1|         43|            74|      1976|          4|female|      113 burns lane|   30.44|   -87.18|          22247.0|      45360.0|   14587.0|         704|               3|
|  2|         48|            64|      1971|          8|  male|  6035 forest avenue|   40.84|   -73.87|          13461.0|

## Crear información de tarjetas por cliente

In [6]:
# Crear resumen de tarjetas

cards_gold = (
    cards_spark
    .groupBy("client_id")
    .agg(
        count("id").alias("total_cards"),
        countDistinct("card_brand").alias("total_card_brands"),
        countDistinct("card_type").alias("total_card_types"),
        round(avg("credit_limit"), 2).alias("average_credit_limit"),
        round(sum("credit_limit"), 2).alias("total_credit_limit")
    )
)

## Crear información de transacciones por cliente

In [7]:
# Crear resumen de transacciones

transactions_gold = (
    transactions_spark
    .groupBy("client_id")
    .agg(
        count("id").alias("total_transactions"),
        round(sum("amount"), 2).alias("total_spent"),
        round(avg("amount"), 2).alias("average_transaction"),
        round(min("amount"), 2).alias("minimum_transaction"),
        round(max("amount"), 2).alias("maximum_transaction"),
        countDistinct("merchant_city").alias("total_merchant_cities")
    )
)

## Crear información financiera por cliente

In [8]:
# Crear información financiera

users_gold = (
    users_spark
    .select(
        "id",
        "current_age",
        "retirement_age",
        "gender",
        "yearly_income",
        "per_capita_income",
        "total_debt",
        "credit_score",
        "num_credit_cards"
    )
)

## Unir la información de clientes

In [9]:
# Unir información

customer_gold = (
    users_gold
    .join(
        cards_gold,
        users_gold.id == cards_gold.client_id,
        "left"
    )
    .drop(cards_gold.client_id)
    .join(
        transactions_gold,
        users_gold.id == transactions_gold.client_id,
        "left"
    )
    .drop(transactions_gold.client_id)
)

## Crear métricas financieras

In [10]:
# Crear métricas financieras

customer_gold = (
    customer_gold
    .withColumn(
        "spending_to_income_ratio",
        round(
            col("total_spent") / col("yearly_income"),
            4
        )
    )
    .withColumn(
        "debt_to_income_ratio",
        round(
            col("total_debt") / col("yearly_income"),
            4
        )
    )
)

## Mostrar información final

In [11]:
# Mostrar información final

customer_gold.show(10)

+---+-----------+--------------+------+-------------+-----------------+----------+------------+----------------+-----------+-----------------+----------------+--------------------+------------------+------------------+-----------+-------------------+-------------------+-------------------+---------------------+------------------------+--------------------+
| id|current_age|retirement_age|gender|yearly_income|per_capita_income|total_debt|credit_score|num_credit_cards|total_cards|total_card_brands|total_card_types|average_credit_limit|total_credit_limit|total_transactions|total_spent|average_transaction|minimum_transaction|maximum_transaction|total_merchant_cities|spending_to_income_ratio|debt_to_income_ratio|
+---+-----------+--------------+------+-------------+-----------------+----------+------------+----------------+-----------+-----------------+----------------+--------------------+------------------+------------------+-----------+-------------------+-------------------+------------

## Guardar GOLD

In [12]:
# Guardar GOLD

customer_gold.write.mode("overwrite").parquet(
    f"{OUTPUT_PATH}/customer_financial_summary.parquet"
)

transactions_gold.write.mode("overwrite").parquet(
    f"{OUTPUT_PATH}/transactions_summary.parquet"
)

cards_gold.write.mode("overwrite").parquet(
    f"{OUTPUT_PATH}/cards_summary.parquet"
)

print("Pipeline GOLD completado")

Pipeline GOLD completado


In [13]:
# Finalizar Spark

spark.stop()